# Эмбеддинги и векторный поиск

Превращаем чанки из урока 3 в векторы фиксированной длины (эмбеддинги),
чтобы искать по документации по смыслу, а не по совпадению слов.

## Пересобираем чанки

Переменная `chunks` жила в `chunking.ipynb` — здесь собираем её заново
той же функцией `chunk_text()`.

In [3]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.

    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются

    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [4]:
from pathlib import Path

DOCS_DIR = Path("docs")

chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

Всего чанков: 530


## Считаем эмбеддинги для чанков

Модель `paraphrase-multilingual-MiniLM-L12-v2` — многоязычная (включая русский),
выдаёт векторы длиной 384. При первом запуске скачается с Hugging Face (~470 МБ),
дальше берётся из локального кеша.

`normalize_embeddings=True` приводит каждый вектор к единичной длине —
тогда косинусная близость эквивалентна скалярному произведению,
и с FAISS будет удобнее работать.

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

/Users/doskhanstybayev/Desktop/Projects/ai_assistant_free_track/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загружаем модель эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches: 100%|██████████| 17/17 [00:02<00:00,  5.83it/s]

Размер матрицы эмбеддингов: (530, 384)


## Заглянем внутрь

Сами числа ничего не говорят — они имеют смысл только в сравнении друг с другом.
Норма равна единице, потому что мы запросили нормализацию векторов.

In [6]:
print(f"Первый чанк: {chunks[0]['text'][:10]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

Первый чанк: # Document...
Его эмбеддинг (первые 10 чисел): [-0.0613074  -0.05510945 -0.03115823 -0.01005449  0.03433183  0.04836935
 -0.08265641  0.02022992 -0.00097118  0.04949648]
Длина вектора (норма): 1.0000


## Строим векторный индекс

Упаковываем эмбеддинги в FAISS-индекс. `IndexFlatIP` — «плоский» индекс
с метрикой Inner Product (скалярное произведение): он сравнивает запрос
со всеми векторами по очереди, без приближений. Для нескольких сотен чанков
этого достаточно.

Так как векторы нормализованы, скалярное произведение и есть косинусная близость.
`index.add()` принимает матрицу (N, dim) и добавляет каждую строку как отдельный вектор.

In [7]:
import faiss

dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"В индексе {index.ntotal} векторов размерности {dim}")

В индексе 530 векторов размерности 384


## Первый поиск

Собираем всё в функцию `vector_search()`. Логика повторяет алгоритм поиска по облаку точек:

1. Превращаем запрос в вектор **той же моделью**, что и чанки.
2. Просим у индекса top-k ближайших.
3. Для каждого попадания возвращаем текст чанка, его источник и оценку близости.

Модель для запроса всегда должна совпадать с моделью для чанков — векторы разных
моделей живут в разных пространствах, и расстояние между ними не имеет смысла.

In [8]:
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    """Возвращает top_k чанков, наиболее близких к запросу по смыслу."""
    # 1. Считаем эмбеддинг запроса
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)

    # 2. Ищем ближайших соседей
    scores, indices = index.search(query_emb, top_k)

    # 3. Собираем результат с метаданными
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

## Проверяем на трёх вопросах

Для каждого вопроса выводим топ-3 чанков с источником и `score`.
Ожидание: топ-1 попадает в тематически правильный раздел документации,
а его `score` заметно выше, чем у второго и третьего.

Второй вопрос задан на русском, а документация на английском —
многоязычная модель эмбеддингов должна справиться.

In [9]:
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    for i, r in enumerate(vector_search(q, top_k=3), 1):
        print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
        print(f"    {r['text'][:120].strip()}...\n")

=== What HTTP method does /api/generate use? ===
#1  [docs/api/introduction.mdx]  score=0.466
    ## Base URL

After installation, Ollama's API is served by default at:

```
http://localhost:11434/api
```

For running...

#2  [docs/faq.mdx]  score=0.461
    ```shell
docker build -t ollama-with-ca .
docker run -d -e HTTPS_PROXY=https://my.proxy.example.com -p 11434:11434 ollam...

#3  [docs/api/introduction.mdx]  score=0.443
    ## Example request

Once Ollama is running, its API is automatically available and can be accessed via `curl`:

```shell...

=== Как сменить папку, где хранятся модели? ===
#1  [docs/faq.mdx]  score=0.586
    ## Where are models stored?

- macOS: `~/.ollama/models`
- Linux: `/usr/share/ollama/.ollama/models`
- Windows: `C:\User...

#2  [docs/windows.mdx]  score=0.581
    To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `...

#3  [docs/modelfile.mdx]  score=0.440
    ```shell
ollama show --mode